<img src="https://udemedellin.edu.co/wp-content/uploads/2022/10/logo_udemedellin2.png" width="30%">

**ESPECIALIZACIÓN EN CIENCIA DE DATOS E INTELIGENCIA ARTIFICIAL**

##  Airbnb Price Prediction Pipeline
### *End-to-End MLOps Architecture with PyCaret & LightGBM*
---

DOCENTE:<br>
Oscar Nicolas Gomez Giraldo<br>

NOMBRES:
1. Santiago Castañeda Garcia
2. Andres Eduardo Medina
3. David Alejandro Montilla Orjuela


Mayo 2026


---

In [17]:
# =========================
# Standard library imports
# =========================
import os
import sys

# =========================
# Third-party imports
# =========================
import hydra
import numpy as np
import pandas as pd
from hydra import compose, initialize
from pycaret.regression import *


from sklearn.compose import (
    ColumnTransformer,
    TransformedTargetRegressor,
)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from pycaret.regression import *
from sklearn.pipeline import Pipeline

In [18]:
# 1. Limpiar cualquier instancia previa de Hydra de forma segura
if hydra.core.global_hydra.GlobalHydra.instance().is_initialized():
    hydra.core.global_hydra.GlobalHydra.instance().clear()

# 2. Inicializar Hydra 
with initialize(version_base=None, config_path="../config"):
    # 3. Cargar la configuración 
    cfg = compose(config_name="config")

# 4. Extraer las columnas desde archivo YAML de Hydra
text_features = list(cfg.features.text)
numeric_features = list(cfg.features.numeric)
binary_features = list(cfg.features.binary)
target = cfg.target

# Combinamos todas las columnas necesarias para no leer el archivo completo 
cols_to_read = text_features + numeric_features + binary_features + [target]

# 5. Leer el DataFrame filtrando solo las columnas del YAML
df_raw = pd.read_csv(
    cfg.data.path,
    usecols=cols_to_read
)


df_raw.head()




,host_is_superhost,neighbourhood_cleansed,property_type,room_type,accommodates,bathrooms,bedrooms,beds,price,guests_included,minimum_nights,number_of_reviews,review_scores_rating,instant_bookable
0,t,Copacabana,Condominium,Entire home/apt,5,1.0,2.0,2.0,$332.00,2,4,243,93.0,t
1,f,Copacabana,Apartment,Entire home/apt,2,1.0,1.0,2.0,$160.00,2,7,235,94.0,f
2,t,Ipanema,Apartment,Entire home/apt,3,1.0,1.0,2.0,$273.00,2,2,271,96.0,t
3,t,Ipanema,Apartment,Entire home/apt,3,1.5,1.0,2.0,$378.00,2,2,169,94.0,f
4,t,Copacabana,Loft,Entire home/apt,2,1.0,1.0,1.0,$130.00,2,3,316,98.0,f


In [19]:
# 1. Añadir la raíz del proyecto al path de Python
sys.path.append(os.path.abspath(".."))

# 2. importar  función 
from src.plots.plots import generar_diccionario

generar_diccionario(df_raw)


-------TABLA CON TIPO DE VARIABLE, VALORES UNICOS, % DE NULOS Y MODA-------



,Variable,Tipo pandas,Cantidad de valores únicos,% Valores faltantes,Valor más frecuente (Moda)
0,host_is_superhost,object,2,0.06,f
1,neighbourhood_cleansed,object,153,0.00,Copacabana
2,property_type,object,37,0.00,Apartment
3,room_type,object,4,0.00,Entire home/apt
4,accommodates,int64,22,0.00,4
5,bathrooms,float64,33,0.16,1.0
6,bedrooms,float64,17,0.12,1.0
7,beds,float64,38,0.14,1.0
8,price,object,856,0.00,$202.00
9,guests_included,int64,21,0.00,1


In [20]:

# 1. Subimos un nivel ("..") para apuntar a la raíz del proyecto y lo añadimos al camino de Python
raiz_proyecto = os.path.abspath(os.path.join(".."))
if raiz_proyecto not in sys.path:
    sys.path.append(raiz_proyecto)

# 2. 
from src.processing.clean_target import clean_raw_data

# 3. Limpiamos el target
df_cleaned = clean_raw_data(df_raw, columna_target=target)
print(" ¡Datos limpiados con éxito!")


# 4. Forzar las columnas que deben ser numéricas a tipo float
print(" Purificando variables numéricas leídas como object...")
for col in numeric_features:
    if col in df_cleaned.columns:
        # errors='coerce' convierte cualquier texto basura o espacio en un NaN numérico limpio
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce').astype(float)

print(" Variables de tipo object convertidas a float correctamente.")

 ¡Datos limpiados con éxito!
 Purificando variables numéricas leídas como object...
 Variables de tipo object convertidas a float correctamente.


In [21]:
# ----------------------------------------------------------------------
#  Filtrar solo las columnas que SÍ existen en el CSV
# ----------------------------------------------------------------------
todas_las_categoricas = text_features + binary_features

# Asegurar de que PyCaret y Sklearn solo busquen columnas reales
categoricas_reales = [col for col in todas_las_categoricas if col in df_cleaned.columns]
numericas_reales = [col for col in numeric_features if col in df_cleaned.columns]


columnas_faltantes = set(todas_las_categoricas + numeric_features) - set(df_cleaned.columns)
if columnas_faltantes:
    print(f"¡Atención! Estas columnas están en el YAML pero NO existen en archivo CSV: {columnas_faltantes}")
else:
    print("Todas las columnas del YAML coinciden con el archivo CSV.")

Todas las columnas del YAML coinciden con el archivo CSV.


In [22]:

from src.processing.clean_target import clean_raw_data
# 1.limpieza (Target)
df_cleaned = clean_raw_data(df_raw, columna_target=target)

# 2. Forzar las columnas que deben ser numéricas a tipo float
for col in numeric_features:
    if col in df_cleaned.columns:
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce').astype(float)

# 3. FILTRO DE ALTA CARDINALIDAD DIRECTO EN PANDAS
print(" Agrupando categorías menores al 0.6% en 'others'...")
threshold = 0.006
other_label = 'others'

# filtrar las variables de texto 
for col in text_features:
    if col in df_cleaned.columns:
        # Calcular la frecuencia relativa de cada valor
        frecuencias = df_cleaned[col].value_counts(normalize=True)
        # Identificamos los valores que SÍ cumplen con el mínimo del 0.6%
        valores_frecuentes = frecuencias[frecuencias >= threshold].index.tolist()
        
        # Si el valor no es frecuente, lo mandamos a 'others'
        df_cleaned[col] = df_cleaned[col].apply(
            lambda x: x if (pd.isna(x) or x in valores_frecuentes) else other_label
        )

print(" ¡Datos reducidos y purificados con éxito!")


 Agrupando categorías menores al 0.6% en 'others'...
 ¡Datos reducidos y purificados con éxito!


### Resumen de Transformadores Personalizados (`transforms.py`)

#### 1. MissingIndicatorTargeted
* **Función:** Detecta columnas con alta tasa de datos faltantes y crea una bandera binaria antes de la imputación.
* **Mecanismo:** En el `.fit()`, identifica variables que superan el umbral de nulos (ej: `threshold=0.20`). En el `.transform()`, genera una nueva columna con el sufijo `_was_missing` asignando `1` (era nulo) o `0` (tenía dato).
* **Propósito de MLOps:** Conserva el patrón de ausencia del dato como información útil para los árboles de LightGBM y almacena los nombres de las columnas (`feature_names_in_`) para asegurar la consistencia si la API recibe datos crudos en producción.

#### 2. CleanBinaryFeatures
* **Función:** Homologa y limpia variables binarias o booleanas con formatos de texto mixtos o inconsistentes.
* **Mecanismo:** En el `.fit()`, filtra las columnas que realmente existen en el dataset. En el `.transform()`, fuerza el casteo a string, elimina espacios y mapea mediante un diccionario estricto múltiples variantes (`'t'/'f'`, `'True'/'False'`, `'1'/'0'`, `'nan'`) a enteros puros (`1` o `0`).
* **Propósito de MLOps:** Actúa como capa de validación defensiva en la API, evitando que formatos de texto mal estructurados introducidos por usuarios rompan el modelo o sean interpretados erróneamente como variables categóricas.

#### 3. CleanAndGroupTextFeatures
* **Función:** Normaliza texto lingüísticamente y controla la alta cardinalidad agrupando categorías minoritarias.
* **Mecanismo:** Usa expresiones regulares y `unicodedata` para limpiar texto (minúsculas, sin acentos ni caracteres especiales). En el `.fit()`, memoriza las categorías con frecuencia relativa $\ge$ al umbral (ej: `0.006`). En el `.transform()`, cualquier texto fuera de esa lista se reasigna a `'otras'`.
* **Propósito de MLOps:** Previene el *Data Leakage* al congelar las categorías válidas en el entrenamiento, evita la explosión dimensional al aplicar *One-Hot Encoding* y mitiga el riesgo de sobreajuste (*overfitting*) en producción.

In [23]:

# Importar las clases personalizadas desde el archivo
from src.processing.transformers import (
    MissingIndicatorTargeted, 
    CleanBinaryFeatures, 
    CleanAndGroupTextFeatures
)

# 1. Listas extraídas desde Hydra
todas_las_categoricas = [col for col in (text_features + binary_features) if col in df_cleaned.columns]
numericas_reales = [col for col in numeric_features if col in df_cleaned.columns]
columnas_binarias = [col for col in binary_features if col in df_cleaned.columns]
columnas_texto = [col for col in text_features if col in df_cleaned.columns]

# 2. Implementar Pipeline
pipeline_personalizado = Pipeline([
    ('indicador_nulos_20', MissingIndicatorTargeted(threshold=0.20)),
    ('limpieza_binarias', CleanBinaryFeatures(binary_columns=columnas_binarias)),
    ('limpieza_y_agrupacion_texto', CleanAndGroupTextFeatures(
        text_columns=columnas_texto, 
        threshold=0.006, 
        other_label='otras'
    ))
])

# 3. Aplicamos log1p directamente a la columna precio
df_cleaned[target] = np.log1p(df_cleaned[target])

# 4. Setup para PyCaret
reg_setup = setup(
    data=df_cleaned, 
    target=target,
    numeric_features=numericas_reales,
    categorical_features=todas_las_categoricas,
    custom_pipeline=pipeline_personalizado, 
    transform_target=False, 
    numeric_imputation='median',         
    categorical_imputation='mode',     
    session_id=42,
    preprocess=True,                    
    verbose=True
)


,Description,Value
0,Session id,42
1,Target,price
2,Target type,Regression
3,Original data shape,"(33708, 14)"
4,Transformed data shape,"(33708, 46)"
5,Transformed train set shape,"(23595, 46)"
6,Transformed test set shape,"(10113, 46)"
7,Numeric features,8
8,Categorical features,5
9,Rows with missing values,45.8%


### Inicialización del Experimento (PyCaret Setup)

Al ejecutar la fase de inicialización mediante `setup()`, PyCaret consolidó el entorno de experimentación bajo los siguientes parámetros analíticos y estructurales:

* **Arquitectura del Dataset:** Se procesó un volumen inicial de **33,708 registros**, aplicando un split estructurado de 70% para el entrenamiento (**23,595 filas**) y 30% para el testeo definitivo (**10,113 filas**).
* **Validación del Pipeline Personalizado (`Custom pipeline: Yes`):** El entorno reconoció con éxito los transformadores modulares nativos (`transforms.py`), ejecutándolos como la capa primaria de curación antes de la fase de modelado. Esto expandió la dimensionalidad de 14 a **46 características operativas**.
* **Tratamiento Crítico de Faltantes:** El sistema detectó que el **45.8% de las filas contenían valores nulos**. Esto validó la necesidad del indicador de ausencia personalizado, resolviendo el remanente mediante imputaciones basadas en la mediana (numéricas) y la moda (categóricas). Esto debido a que en el edad se visualizo que las variables presentan una distribución asimétrica positiva (right-skewed).
* **Estrategia de Validación Robusta:** Se configuró un esquema de **10-Fold Cross Validation**. Esta metodología garantiza que las métricas de evaluación de los algoritmos no dependan de una partición aleatoria favorable, dotando de certeza estadística a los resultados de producción.

In [ ]:
# Comparamos todos los modelos de regresión disponibles
modelo_ganador = compare_models()

,,
,,
Initiated,. . . . . . . . . . . . . . . . . .,09:52:26
Status,. . . . . . . . . . . . . . . . . .,Fitting 10 Folds
Estimator,. . . . . . . . . . . . . . . . . .,Random Forest Regressor


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
knn,K Neighbors Regressor,0.5643,0.5600,0.7480,0.4791,0.1093,0.1002,0.7410
ridge,Ridge Regression,0.5851,0.6354,0.7924,0.4076,0.1105,0.1026,0.2330
br,Bayesian Ridge,0.5851,0.6356,0.7924,0.4074,0.1105,0.1026,0.2770
lr,Linear Regression,0.5854,0.6358,0.7926,0.4072,0.1105,0.1026,1.5310
huber,Huber Regressor,0.6039,0.7186,0.8404,0.3294,0.1205,0.1054,1.0070
omp,Orthogonal Matching Pursuit,0.6787,0.7984,0.8917,0.2553,0.1268,0.1200,0.2960
dt,Decision Tree Regressor,0.6631,0.8041,0.8965,0.2514,0.1296,0.1160,0.3830
en,Elastic Net,0.7154,0.8383,0.9154,0.2196,0.1322,0.1268,0.2980
lasso,Lasso Regression,0.7645,0.9373,0.9680,0.1281,0.1404,0.1354,0.9010
llar,Lasso Least Angle Regression,0.7645,0.9373,0.9680,0.1281,0.1404,0.1354,0.2600


Processing:   0%|          | 0/77 [00:00<?, ?it/s]

### Justificación Técnica del Modelo Seleccionado

Tras evaluar múltiples algoritmos mediante la función de comparación automatizada de PyCaret, **LightGBM (Light Gradient Boosting Machine)** fue seleccionado como el modelo de producción debido a su superioridad técnica y de cómputo:

| Modelo | MAE* | RMSE* | $R^2$ | MAPE | Tiempo Entrenamiento (Seg) |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **LightGBM** | **0.4875** | **0.6531** | **0.6029** | **8.55%** | **0.7500** |
| Gradient Boosting | 0.5071 | 0.6741 | 0.5770 | 8.91% | 1.2610 |

*> **Nota sobre las métricas:** Debido a que la variable objetivo fue transformada a escala logarítmica para estabilizar la varianza y mitigar el efecto de los valores atípicos, las métricas de error (MAE y RMSE) están expresadas en unidades logarítmicas.*

* **Poder de Explicación:** Con un **$R^2$ de 60.27%**, el modelo asimila la mayor parte de la variabilidad de los precios del mercado a partir de las características procesadas en el pipeline personalizado.
error porcentual medio (**MAPE**) de apenas el **8.55%**, esto es que nuestro modelo estima con un 8.5% el valor por encima o por debajo del valor del alojamiento.

En nuestro caso, el MAE (**0.48**) y el RMSE (**0.65**) se encuentran notablemente cerca. Esta estabilidad matemática demuestra que **el modelo es altamente confiable en producción**, validando que la estrategia de agrupar la alta cardinalidad en el pipeline impidió que el algoritmo generara predicciones absurdas o descontroladas.

In [ ]:
# Entrenamos el LightGBM con absolutamente todos los datos
modelo_final_lgb = finalize_model(modelo_ganador)

In [ ]:
# Muestra la importancia y relación de todas las características con el precio
plot_model(modelo_final_lgb, plot='feature_all')

### Importancia de Características (Feature Importance)

Para entender cómo el modelo **LightGBM** calcula las predicciones de precios, se analizó el impacto individual de cada variable en los árboles de decisión. A continuación, se detallan los hallazgos principales basados en el comportamiento del mercado:

* **Estrategia de Reserva (`minimum_nights`):** Se consolidó como la variable de mayor impacto. Esto se alinea con las dinámicas de negocio de Airbnb, donde las estancias mínimas prolongadas (enfoque corporativo/mensual) aplican descuentos agresivos frente a las tarifas volátiles por noche de estancias cortas.
* **Infraestructura Crítica (`bathrooms`):** El número de baños demostró tener mayor relevancia predictiva que el conteo de habitaciones o camas individuales, actuando como el principal diferenciador de comodidad y segmentación de precios en el dataset.
* **Validación Social (`number_of_reviews`):** La antigüedad y volumen de actividad del anuncio interactúan fuertemente con la predicción, capturando el factor de confianza y posicionamiento del inmueble en la plataforma.
* **Premium Geográfico (`Barra da Tijuca`):** El modelo logró aislar geográficamente el impacto del vecindario, identificando de manera autónoma a zonas de alta plusvalía como *Barra da Tijuca* como un catalizador directo del incremento en el valor de la predicción.

In [ ]:
# 1. Gráfico de Error (Predicho vs. Real) -
plot_model(modelo_final_lgb, plot='error')

### Análisis del Error de Predicción (Prediction Error Plot)

Para validar el comportamiento del modelo final frente a datos nunca antes vistos, se evaluó el comportamiento de los residuos en el set de prueba (*Hold-out*):

* **Consistencia Predictiva ($R^2 = 0.638$):** El modelo retiene y eleva su capacidad explicativa en la fase de testeo, alcanzando un **63.8%** de precisión general. Esto confirma la ausencia de sobreajuste (*overfitting*) y valida la solidez de las etapas de ingeniería de características previas.
* **Alineación en Mercado Masivo:** La alta densidad de registros concentrada en los rangos centrales refleja que la línea de mejor ajuste (`best fit`) se acopla casi simétricamente a la línea de identidad ideal. Esto garantiza predicciones óptimas en el segmento comercial estándar de la plataforma.
* **Efecto de Regresión a la Media:** Se observa una sutil subestimación en propiedades de ultra-lujo (valores logarítmicos > 8.5). Este comportamiento es típico y seguro en modelos basados en árboles de decisión (LightGBM), ya que el algoritmo prioriza la varianza central del mercado para evitar predicciones desproporcionadas en entornos de producción.

In [ ]:
# 2. Gráfico de Residuos (Para ver dónde se equivoca más el modelo)
plot_model(modelo_final_lgb, plot='residuals')

###  Análisis de Residuos (Residuals Plot)

Para validar los supuestos fundamentales del modelo lineal basado en árboles (LightGBM), se analizó la distribución de los residuos en los conjuntos de entrenamiento y prueba:

* **Consistencia Metodológica:** El $R^2$ de entrenamiento (**0.640**) frente al de prueba (**0.638**) exhibe una convergencia casi perfecta. Esta paridad descarta problemas de sobreajuste (*overfitting*) o subajuste (*underfitting*), asegurando la reproducibilidad del sistema.
* **Homocedasticidad Controlada:** La dispersión de los errores se concentra de manera uniforme y simétrica alrededor de la línea de referencia $0$ en el rango central de predicción (escala logarítmica de 5 a 8), demostrando estabilidad en el cálculo de tarifas del mercado masivo.
* **Normalidad de los Errores:** El histograma de frecuencias lateral confirma que los residuos siguen una **distribución normal centrada en cero**. Esto valida estadísticamente que los errores (residuales) del modelo se deben a la volatilidad aleatoria intrínseca del negocio de Airbnb y no a sesgos algorítmicos o fallos de especificación en el preprocesamiento.

In [ ]:
# 3. Curva de Aprendizaje (Para ver si falta aprender o si hay overfitting)
plot_model(modelo_final_lgb, plot='learning')

### Curva de Aprendizaje (Learning Curve)

Para auditar la eficiencia del volumen de datos y diagnosticar el balance entre sesgo y varianza (*Bias-Variance Tradeoff*), se evaluó el comportamiento de las puntuaciones de entrenamiento frente a la validación cruzada:

* **Estabilización de la Generalización:** A medida que el volumen de registros aumenta hacia las 20,000 instancias, la curva de entrenamiento (`Training Score`) y la de validación (`Cross Validation Score`) convergen de forma asintótica, estabilizándose en un estrecho margen de error. Esto confirma matemáticamente la **ausencia de sobreajuste (Overfitting)**.
* **Optimización del Volumen de Datos:** El aplanamiento o "meseta" de la curva verde al superar los 18,000 registros demuestra que el algoritmo LightGBM alcanzó su máxima capacidad de aprendizaje con el set de características actual. Esto valida que el tamaño del dataset es óptimo y que mejoras futuras dependerían de la incorporación de nuevas variables de negocio (*Feature Generation*) y no de la simple recolección de más registros.

In [ ]:
# Muestra una tabla detallada con todos los hiperparámetros del modelo ganador
plot_model(modelo_final_lgb, plot='parameter')

###  Arquitectura e Hiperparámetros del Modelo Final (LightGBM)

El modelo de producción final basado en **Light Gradient Boosting Machine** quedó configurado con los siguientes parámetros analíticos y estructurales óptimos:

* **Estrategia de Optimización (`boosting_type: gbdt`):** Utiliza *Gradient Boosting Decision Tree* tradicional. Es el núcleo algorítmico que construye árboles secuenciales minimizando los errores (residuos) de los árboles anteriores de forma iterativa.
* **Ritmo de Aprendizaje (`learning_rate: 0.1`):** Define el tamaño del paso que da el modelo al corregir errores en cada iteración. Un valor de `0.1` provee un balance óptimo: es lo suficientemente bajo para evitar el sobreajuste (*overfitting*) y lo suficientemente alto para converger de forma eficiente sin requerir miles de árboles.
* **Profundidad y Complejidad del Árbol (`max_depth: -1` y `num_leaves: 31`):** * Un `max_depth` de `-1` significa que **no hay límite estricto de profundidad vertical** para los árboles.
  * En su lugar, LightGBM controla la complejidad mediante `num_leaves: 31` (máximo 31 hojas por árbol) creciendo de forma asimétrica basada en las hojas con mayor pérdida (*leaf-wise*). Esta es la razón principal de su alta precisión frente al Gradient Boosting tradicional.
* **Control de Ruido y Regularización (`min_child_samples: 20`):** Exige que cada hoja terminal contenga como mínimo **20 registros** para poder generar una división. Esto actúa como un freno de seguridad contra el ruido estadístico, impidiendo que el modelo cree reglas ultraespecíficas para alojamientos aislados.
* **Capacidad de Cómputo (`n_estimators: 100`):** El modelo se consolida mediante un ensamble secuencial de **100 árboles de decisión**.
* **Eficiencia en Producción (`n_jobs: -1`):** Configurado para usar **todos los núcleos de procesamiento disponibles en paralelo** en el servidor, garantizando que el tiempo de entrenamiento se reduzca al mínimo absoluto (`0.75` segundos) y que la API responda en milisegundos en tiempo real.

In [ ]:
# 1. definir X_train y y_train 
X_train_pycaret = get_config('X_train')
y_train_pycaret = get_config('y_train')

# 2. revertir el logaritmo de forma nativa
modelo_empaquetado_api = TransformedTargetRegressor(
    regressor=modelo_final_lgb,  # <--- Modelo final entrenado
    func=np.log1p,         
    inverse_func=np.expm1  
)

# 3. Entrenar el modelo empaquetado usando los datos extraídos de PyCaret
modelo_empaquetado_api.fit(X_train_pycaret, y_train_pycaret)

# 4. Crear la carpeta 'app' si no existe
os.makedirs("app", exist_ok=True)



In [ ]:
# 1. Obtener la ruta actual 
ruta_actual = os.getcwd()

# 2. Subir un nivel a la raíz (Proyecto_3) y apuntar a la carpeta 'app' 
ruta_raiz = os.path.abspath(os.path.join(ruta_actual, '..'))
ruta_api_app_real = os.path.join(ruta_raiz, 'app')

# Asegurar que la carpeta exista en la raíz
os.makedirs(ruta_api_app_real, exist_ok=True)

# 3. Crear el camino completo hacia el archivo (SIN el .pkl al final)
camino_final_modelo = os.path.join(ruta_api_app_real, 'pipeline_ganador')

# 4. Guardar con PyCaret en la ubicación correcta
save_model(modelo_final_lgb, camino_final_modelo)

print(f" ¡Modelo guardado con éxito en la ubicación REAL de la API!")
print(f" {camino_final_modelo}.pkl")

## Conclusiones del Proyecto

El desarrollo y despliegue de esta API REST representa una solución de ingeniería de extremo a extremo, consolidando las siguientes conclusiones clave:

### 1. Ingeniería de Características Modular y Pipeline Personalizado
* **Arquitectura Orientada a Objetos:** Se diseñó un flujo de preparación robusto extendiendo las capacidades de Scikit-Learn mediante un **Pipeline personalizado**. Esto permitió encapsular transformadores propios (`MissingIndicatorTargeted`, `CleanBinaryFeatures` y `CleanAndGroupTextFeatures`), abstrayendo la complejidad de la preparación de datos.
* **Reducción Automatizada de Alta Cardinalidad:** El transformador `CleanAndGroupTextFeatures` resolvió de forma elegante la dispersión en variables categóricas (como barrios). Al agrupar dinámicamente las categorías con frecuencia menor al 0.6% bajo la etiqueta `'otras'`, se previno la explosión dimensional (*One-Hot Encoding*) y el sobreajuste (*overfitting*), blindando estadísticamente al modelo.
* **Purificación Defensiva:** El pipeline gestiona de forma nativa la identificación de datos nulos y la consistencia en variables binarias, asegurando un filtrado homogéneo antes de la fase de modelado con PyCaret.

### 2. Ciencia de Datos y Modelado (`LightGBM` + `PyCaret`)
* **Eficiencia Algorítmica:** El uso de **LightGBM**, optimizado mediante PyCaret, demostró un rendimiento superior y una excelente capacidad de generalización frente a modelos tradicionales, capturando eficazmente las relaciones no lineales del inmueble.
* **Estabilización de Varianza:** La transformación logarítmica de la variable objetivo en el entrenamiento mitigó la heterocedasticidad, estabilizando los residuos del modelo para predicciones más precisas en rangos monetarios normales.

### 3. Ingeniería de Software y Despliegue en Producción (`FastAPI`)
* **Portabilidad Absoluta sin Desfase (Feature Drift):** Al empaquetar de forma secuencial la curación avanzada y el modelado dentro del pipeline binario `.pkl`, se garantizó que la API aplique exactamente las mismas transformaciones a los datos en caliente de producción que las usadas en la etapa de experimentación, eliminando el riesgo de *Data Leakage*.
* **Robustez y Validación:** La combinación de FastAPI y Pydantic valida estrictamente los esquemas de entrada (`PredictionInput`), mientras que el Pipeline procesa internamente la información y ejecuta la transformación matemática inversa (`np.expm1`) para devolver predicciones limpias y en valores monetarios reales.

### 4. Impacto de Negocio y Mantenibilidad (`Poetry` + `Git`)
* **Toma de Decisiones Automatizada:** La solución provee una herramienta de *Dynamic Pricing* competitiva para que nuevos anfitriones maximicen sus tasas de ocupación basándose en la realidad analítica del mercado local.
* **Gobernanza del Proyecto:** El aislamiento de dependencias con **Poetry** y la exclusión de datos crudos pesados mediante un archivo `.gitignore` estricto aseguran un código fuente ligero, replicable y alineado con las mejores prácticas de desarrollo.